# Verificacion del Deployment del Sistema LADA

Este notebook verifica que el modelo guardado se puede cargar y utilizar correctamente para hacer predicciones.

## Carga del Modelo

In [2]:
import pickle
import pandas as pd
import numpy as np
import sys

sys.path.append('./models')
from lada_funciones import predecir_probabilidad_exito

In [3]:
with open('./models/lada_modelo.pkl', 'rb') as f:
    modelo = pickle.load(f)

resultados_por_nivel = modelo['resultados_por_nivel']
df_estudiantes = modelo['df_estudiantes']
df_train = modelo['df_train']
metadata = modelo['metadata']

print(f"Modelo version {metadata['version']} cargado")
print(f"Fecha entrenamiento: {metadata['fecha_entrenamiento']}")
print(f"Estudiantes en base de datos: {len(df_estudiantes):,}")

Modelo version 1.0 cargado
Fecha entrenamiento: 2025-11-30 06:44:50
Estudiantes en base de datos: 99,985


## Metricas del Modelo

In [4]:
metricas = metadata['metricas']

print("Metricas de evaluacion:")
print(f"  MAE: {metricas['mae']:.4f}")
print(f"  RMSE: {metricas['rmse']:.4f}")
print(f"  Accuracy ±10%: {metricas['accuracy_10']:.2%}")
print(f"  Accuracy ±20%: {metricas['accuracy_20']:.2%}")
print(f"  Correlacion: {metricas['correlacion']:.4f}")

Metricas de evaluacion:
  MAE: 0.0912
  RMSE: 0.1839
  Accuracy ±10%: 77.35%
  Accuracy ±20%: 87.10%
  Correlacion: 0.2511


## Prediccion 1: Estudiante Existente

In [ ]:
estudiante_1 = df_estudiantes.iloc[10]['CODIGO_ESTUDIANTE']
pga_1 = df_estudiantes.iloc[10]['PGA']

perfil_1 = {
    'estudiante_id': estudiante_1,
    'cursos': ['CRS_00017889', 'CRS_00017890', 'CRS_00017891'],
    'num_cursos': 3,
    'creditos': 9
}

#El PGA se incluye en el dataframe de estudiantes. No es necesario pasarlo por separado.

resultado_1 = predecir_probabilidad_exito(
    perfil_1,
    None,
    df_estudiantes,
    resultados_por_nivel
)

print(f"Estudiante: {estudiante_1}")
print(f"PGA actual: {pga_1:.2f}")
print(f"Cursos planificados: {perfil_1['num_cursos']}")
print(f"\nResultado:")
print(f"  Probabilidad de exito: {resultado_1['probabilidad_exito']*100:.1f}%")
print(f"  Nivel usado: {resultado_1['nivel_usado']}") # El nivel de especificidad. 
                                                      # Nivel 1 -> Que sea estudiand(opcional), 
                                                      # Nivel 3 -> Que tenga exactamente los mismos cursos 
print(f"  Confianza: {resultado_1['confianza']}")
print(f"  Cluster: {resultado_1['cluster_id']}") #A partr de los niveles de espicificidad (['nivel_usado']) con el PGA usted le asigan un cluster al estudiante. 
                                                 #El número de cluster es el ID del cluster asignado.
print(f"  Estudiantes similares: {resultado_1['num_estudiantes_similares']}") # Número de estudiantes similares usados para hacer el cluster.
                                                                              # Hay más o menos 7000 por cluster.

Estudiante: EST_00000045
PGA actual: 4.36
Cursos planificados: 3

Resultado:
  Probabilidad de exito: 97.0%
  Nivel usado: NIVEL_2
  Confianza: ALTA
  Cluster: 0
  Estudiantes similares: 6898


## Prediccion 2: Estudiante con Alto PGA

In [ ]:
estudiantes_alto_pga = df_estudiantes[df_estudiantes['PGA'] >= 4.55]
estudiante_2 = estudiantes_alto_pga.iloc[0]['CODIGO_ESTUDIANTE']
pga_2 = estudiantes_alto_pga.iloc[0]['PGA']

perfil_2 = {
    'estudiante_id': estudiante_2,
    'cursos': ['CRS_00017889', 'CRS_00017890', 'CRS_00017891', 'CRS_00017892'], # La prob acaba siendo el "no retirar ninguno"
    'num_cursos': 4,
    'creditos': 12
}

resultado_2 = predecir_probabilidad_exito(
    perfil_2,
    None,
    df_estudiantes,
    resultados_por_nivel
)

print(f"Estudiante: {estudiante_2}")
print(f"PGA actual: {pga_2:.2f}")
print(f"Cursos planificados: {perfil_2['num_cursos']}")
print(f"\nResultado:")
print(f"  Probabilidad de exito: {resultado_2['probabilidad_exito']*100:.1f}%")
print(f"  Nivel usado: {resultado_2['nivel_usado']}")
print(f"  Confianza: {resultado_2['confianza']}")
print(f"  Cluster: {resultado_2['cluster_id']}")
print(f"  Estudiantes similares: {resultado_2['num_estudiantes_similares']}")

Estudiante: EST_00000072
PGA actual: 4.60
Cursos planificados: 4

Resultado:
  Probabilidad de exito: 95.2%
  Nivel usado: NIVEL_2
  Confianza: ALTA
  Cluster: 0
  Estudiantes similares: 5864


## Prediccion 3: Estudiante Nuevo (No en BD)

In [15]:
perfil_3 = {
    'estudiante_id': 'EST_NUEVO_2024',
    'cursos': ['CRS_00017889', 'CRS_00017890'],
    'num_cursos': 2,
    'creditos': 6
}


resultado_3 = predecir_probabilidad_exito(
    perfil_3,
    None,
    df_estudiantes,
    resultados_por_nivel
)

print(f"Estudiante: {perfil_3['estudiante_id']} (nuevo)")
print(f"Cursos planificados: {perfil_3['num_cursos']}")
print(f"\nResultado:")
print(f"  Probabilidad de exito: {resultado_3['probabilidad_exito']*100:.1f}%")
print(f"  Nivel usado: {resultado_3['nivel_usado']}")
print(f"  Confianza: {resultado_3['confianza']}")
print(f"  Cluster: {resultado_3['cluster_id']}")
print(f"  Estudiantes similares: {resultado_3['num_estudiantes_similares']}")

Estudiante: EST_NUEVO_2024 (nuevo)
Cursos planificados: 2

Resultado:
  Probabilidad de exito: 96.1%
  Nivel usado: NIVEL_2
  Confianza: ALTA
  Cluster: 1
  Estudiantes similares: 5515


## Prediccion 4: Diferentes Cantidades de Cursos

In [7]:
estudiante_test = df_estudiantes.iloc[50]['CODIGO_ESTUDIANTE']
pga_test = df_estudiantes.iloc[50]['PGA']

print(f"Estudiante: {estudiante_test} (PGA: {pga_test:.2f})")
print(f"\nProbando diferentes cargas academicas:\n")

escenarios = [
    (['CRS_00017889'], 1),
    (['CRS_00017889', 'CRS_00017890'], 2),
    (['CRS_00017889', 'CRS_00017890', 'CRS_00017891'], 3),
    (['CRS_00017889', 'CRS_00017890', 'CRS_00017891', 'CRS_00017892'], 4),
    (['CRS_00017889', 'CRS_00017890', 'CRS_00017891', 'CRS_00017892', 'CRS_00017893'], 5)
]

for cursos, num in escenarios:
    perfil = {
        'estudiante_id': estudiante_test,
        'cursos': cursos,
        'num_cursos': num,
        'creditos': num * 3
    }
    
    resultado = predecir_probabilidad_exito(
        perfil,
        None,
        df_estudiantes,
        resultados_por_nivel
    )
    
    print(f"{num} curso(s): {resultado['probabilidad_exito']*100:.1f}% | "
          f"{resultado['nivel_usado']} | "
          f"Confianza: {resultado['confianza']}")

Estudiante: EST_00000162 (PGA: 4.61)

Probando diferentes cargas academicas:

1 curso(s): 96.8% | NIVEL_2 | Confianza: ALTA
2 curso(s): 96.1% | NIVEL_2 | Confianza: ALTA
3 curso(s): 97.0% | NIVEL_2 | Confianza: ALTA
4 curso(s): 95.2% | NIVEL_2 | Confianza: ALTA
5 curso(s): 96.5% | NIVEL_2 | Confianza: ALTA


## Analisis de Niveles Jerarquicos

In [9]:
print("Estructura del modelo por nivel:\n")

for nivel in ['NIVEL_1', 'NIVEL_2', 'NIVEL_3']:
    num_firmas = len(resultados_por_nivel[nivel])
    print(f"{nivel}:")
    print(f"  Firmas unicas: {num_firmas:,}")
    
    if num_firmas > 0:
        primera_firma = list(resultados_por_nivel[nivel].keys())[0]
        info = resultados_por_nivel[nivel][primera_firma]
        print(f"  Ejemplo - Clusters: {info['n_clusters']}, Casos: {info['total_casos']}")
    print()

Estructura del modelo por nivel:

NIVEL_1:
  Firmas unicas: 14
  Ejemplo - Clusters: 5, Casos: 113516

NIVEL_2:
  Firmas unicas: 14
  Ejemplo - Clusters: 5, Casos: 113516

NIVEL_3:
  Firmas unicas: 100
  Ejemplo - Clusters: 5, Casos: 2564



## Comparacion de Predicciones por PGA

In [10]:
cursos_fijos = ['CRS_00017889', 'CRS_00017890', 'CRS_00017891']

rangos_pga = [
    ('Bajo', df_estudiantes[df_estudiantes['PGA'] < 3.0].iloc[0]),
    ('Medio', df_estudiantes[(df_estudiantes['PGA'] >= 3.0) & (df_estudiantes['PGA'] < 3.5)].iloc[0]),
    ('Alto', df_estudiantes[df_estudiantes['PGA'] >= 3.5].iloc[0])
]

print("Predicciones segun PGA (mismos cursos):\n")

for categoria, estudiante_row in rangos_pga:
    est_id = estudiante_row['CODIGO_ESTUDIANTE']
    est_pga = estudiante_row['PGA']
    
    perfil = {
        'estudiante_id': est_id,
        'cursos': cursos_fijos,
        'num_cursos': 3,
        'creditos': 9
    }
    
    resultado = predecir_probabilidad_exito(
        perfil,
        None,
        df_estudiantes,
        resultados_por_nivel
    )
    
    print(f"PGA {categoria} ({est_pga:.2f}):")
    print(f"  Probabilidad: {resultado['probabilidad_exito']*100:.1f}%")
    print(f"  Cluster: {resultado['cluster_id']}")
    print(f"  Confianza: {resultado['confianza']}")
    print()

Predicciones segun PGA (mismos cursos):

PGA Bajo (1.50):
  Probabilidad: 55.3%
  Cluster: 1
  Confianza: ALTA

PGA Medio (3.38):
  Probabilidad: 90.4%
  Cluster: 4
  Confianza: ALTA

PGA Alto (4.29):
  Probabilidad: 97.0%
  Cluster: 0
  Confianza: ALTA



## Resumen de Verificacion

In [11]:
print("Sistema LADA - Estado del Deployment\n")
print(f"Modelo: {metadata['version']}")
print(f"Entrenado: {metadata['fecha_entrenamiento']}")
print(f"\nComponentes cargados:")
print(f"  Estudiantes en BD: {len(df_estudiantes):,}")
print(f"  Registros de entrenamiento: {len(df_train):,}")
print(f"  Niveles jerarquicos: 3")
print(f"\nEstado: Operacional")
print(f"\nEl sistema puede:")
print(f"  - Predecir estudiantes existentes")
print(f"  - Predecir estudiantes nuevos")
print(f"  - Manejar diferentes cantidades de cursos")
print(f"  - Seleccionar nivel jerarquico automaticamente")

Sistema LADA - Estado del Deployment

Modelo: 1.0
Entrenado: 2025-11-30 06:44:50

Componentes cargados:
  Estudiantes en BD: 99,985
  Registros de entrenamiento: 535,134
  Niveles jerarquicos: 3

Estado: Operacional

El sistema puede:
  - Predecir estudiantes existentes
  - Predecir estudiantes nuevos
  - Manejar diferentes cantidades de cursos
  - Seleccionar nivel jerarquico automaticamente
